# VSPAERO pitch-trimmed polar example

このNotebookでは、OpenVSPで読み込んだ`.vsp3`モデルに対して`vsp_trimmed_sweep()`を実行し、迎角ごとに

$$
C_{My,\mathrm{tot}}=0
$$

となるエレベータ舵角を求めた滑空極を作成する。

処理の流れは次のとおりである。

1. G103Aの`.vsp3`モデルを読み込む。
2. `WingGeom`と、`Control Group Angles`で有効化された既存の`ELEVATOR_GROUP`を確認する。
3. 少数の迎角についてpitch-trimmed polarを計算する。
4. エレベータ舵角、`CMytot`残差、反復回数、計算時間を確認する。
5. 未補正の滑空性能を表とグラフで確認する。
6. 必要な場合だけ`make_CDo_correction()`を適用する。
7. 結果をCSVとPNGへ保存する。

この計算は`.stab`線形空力モデルを使う`example_turn_trim.ipynb`とは異なり、各評価点でVSPAEROを直接実行する。そのため、OpenVSP Python環境とVSPAERO実行環境が必要であり、迎角点数とトリム評価回数に応じて計算時間が増える。

本Notebookでは、`vsp_trimmed_sweep()` を使って迎角ごとのピッチトリムを直接計算する。

## 1. importとリポジトリ位置

Notebookの実行場所から親ディレクトリを順に調べ、`src/AnalysisVSPAERO.py`を含むリポジトリルートを見つける。

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "AnalysisVSPAERO.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing "
        "src/AnalysisVSPAERO.py."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import openvsp as vsp  # noqa: E402
from src.AnalysisVSPAERO import (  # noqa: E402
    make_CDo_correction,
    vsp_trimmed_sweep,
)

print("repo_root:", repo_root)
print("OpenVSP version:", vsp.GetVSPVersion())

## 2. 入力モデルと解析条件

通常変更する条件をこのセルへまとめる。

最初から多数の迎角を指定すると、各迎角で複数回VSPAEROが実行される。exampleの既定値は3点だけとし、モデル、Control Surface、トリム収束を確認してから迎角列を増やす。

`mass`は質量`kg`である。公開APIでは互換性のため引数名が`Weight`のまま残っている。

In [ ]:
model_name = "G103A"
model_path = (
    repo_root
    / "examples"
    / "models"
    / model_name
    / f"{model_name}.vsp3"
)
output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "trimmed_polar"
    / "results"
)
output_dir.mkdir(parents=True, exist_ok=True)

mass = 580.0
altitude = 0.0
dT = 0.0

# 最初は少数点でモデル設定とトリム収束を確認する。
# alpha_list = [2.0, 4.0, 6.0]

# 本格的なpolarを計算するときの例。
alpha_list = [
    -2.0, -1.5, -1.0, -0.5, 0.0, 0.5,
    1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 10.0, 12.0,
]

elevator_bounds = (-20.0, 20.0)
cmy_tolerance = 5.0e-4
max_trim_evaluations = 30
trim_step_deg = 1.0

# Noneは、読み込んだOpenVSPモデルまたはOpenVSP既定値を維持する。
analysis_method = None
ncpu = None
wake_num_iter = None
fixed_wake_flag = None
verbose = 1

# 抵抗補正はVSPAEROトリム計算とは別の後処理である。
apply_cdo_correction = False
cdo_correction = {
    "xTr": (0.5, 0.7),
    "CDpCL": 0.0016,
    "thickness": 0.19,
    "interference_factor": 1.14,
}

if not model_path.exists():
    raise FileNotFoundError(model_path)

input_summary = pd.Series(
    {
        "model_path": str(model_path),
        "output_dir": str(output_dir),
        "mass_kg": mass,
        "altitude_m": altitude,
        "dT_K": dT,
        "alpha_list_deg": alpha_list,
        "elevator_bounds_deg": elevator_bounds,
        "cmy_tolerance": cmy_tolerance,
        "max_trim_evaluations": max_trim_evaluations,
        "trim_step_deg": trim_step_deg,
        "ncpu": ncpu,
        "wake_num_iter": wake_num_iter,
        "fixed_wake_flag": fixed_wake_flag,
        "apply_cdo_correction": apply_cdo_correction,
    },
    name="value",
)
display(input_summary)

## 3. OpenVSPモデルの読込と前提確認

`vsp_trimmed_sweep()`は、次のモデル構成を前提とする。

- `WingGeom`がちょうど1個存在する。
- `ELEVATOR_GROUP`がちょうど1個存在する。
- `ELEVATOR_GROUP`に有効なControl Surfaceが登録されている。
- `Control Group Angles`で`ELEVATOR_GROUP`がONになっている。
- `VSPAEROSettings`にエレベータの`DeflectionAngle`が存在する。

ここでは解析前にその状態を表示する。Control Surface Groupやgainを作り直す処理はNotebookへ追加しない。

In [ ]:
vsp.ClearVSPModel()
vsp.Update()
vsp.ReadVSPFile(str(model_path))
vsp.Update()

wing_ids = list(vsp.FindGeomsWithName("WingGeom"))
if len(wing_ids) != 1:
    raise RuntimeError(
        "Pitch-trimmed polar requires exactly one WingGeom. "
        f"found={wing_ids}"
    )

control_group_names = [
    vsp.GetVSPAEROControlGroupName(group_index)
    for group_index in range(vsp.GetNumControlSurfaceGroups())
]
elevator_group_indices = [
    group_index
    for group_index, group_name in enumerate(control_group_names)
    if group_name == "ELEVATOR_GROUP"
]
if len(elevator_group_indices) != 1:
    raise RuntimeError(
        "Pitch trim requires exactly one existing ELEVATOR_GROUP. "
        f"found={len(elevator_group_indices)}"
    )

elevator_group_index = elevator_group_indices[0]
active_elevator_surfaces = list(
    vsp.GetActiveCSNameVec(elevator_group_index)
)
if not active_elevator_surfaces:
    raise RuntimeError("ELEVATOR_GROUP has no active control surfaces.")

settings_id = vsp.FindContainer("VSPAEROSettings", 0)
if not settings_id:
    raise RuntimeError("VSPAEROSettings container was not found.")

elevator_group_parm_group = (
    f"ControlSurfaceGroup_{elevator_group_index}"
)
elevator_active_parm_id = vsp.FindParm(
    settings_id,
    "ActiveFlag",
    elevator_group_parm_group,
)
if (
    not elevator_active_parm_id
    or str(elevator_active_parm_id).upper() == "NONE"
):
    raise RuntimeError(
        "Could not find ActiveFlag for ELEVATOR_GROUP."
    )
if not bool(vsp.GetParmVal(elevator_active_parm_id)):
    raise RuntimeError(
        "ELEVATOR_GROUP is disabled in VSPAERO Control Group Angles."
    )

elevator_parm_id = vsp.FindParm(
    settings_id,
    "DeflectionAngle",
    elevator_group_parm_group,
)
if not elevator_parm_id or str(elevator_parm_id).upper() == "NONE":
    raise RuntimeError(
        "Could not find DeflectionAngle for ELEVATOR_GROUP."
    )

compgeom_name = "VSPAEROComputeGeometry"
vsp.SetAnalysisInputDefaults(compgeom_name)
compgeom_inputs = set(vsp.GetAnalysisInputNames(compgeom_name))
thick_geom_set = (
    int(vsp.GetIntAnalysisInput(compgeom_name, "GeomSet")[0])
    if "GeomSet" in compgeom_inputs
    else None
)
thin_geom_set = (
    int(vsp.GetIntAnalysisInput(compgeom_name, "ThinGeomSet")[0])
    if "ThinGeomSet" in compgeom_inputs
    else None
)

model_summary = pd.Series(
    {
        "vsp3_path": vsp.GetVSPFileName(),
        "wing_id": wing_ids[0],
        "control_group_names": control_group_names,
        "elevator_group_index": elevator_group_index,
        "active_elevator_surfaces": active_elevator_surfaces,
        "elevator_group_angle_enabled": bool(
            vsp.GetParmVal(elevator_active_parm_id)
        ),
        "initial_elevator_deg": float(vsp.GetParmVal(elevator_parm_id)),
        "compute_geometry_GeomSet": thick_geom_set,
        "compute_geometry_ThinGeomSet": thin_geom_set,
    },
    name="value",
)
display(model_summary)

## 4. pitch-trimmed polarの計算

各迎角について、最初に揚力係数から飛行速度、Mach数、Reynolds数を求め、その条件でエレベータ舵角を変化させる。

受理条件は、

$$
\left|C_{My,\mathrm{tot}}\right|
\leq
\texttt{cmy\_tolerance}
$$

である。

前の迎角で得たエレベータ舵角と局所勾配を次の迎角の初期値として引き継ぐため、通常は後続caseほど少ない評価回数で収束する。

In [ ]:
trimmed_polar = vsp_trimmed_sweep(
    vsp=vsp,
    alpha_list=alpha_list,
    Weight=mass,
    altitude=altitude,
    dT=dT,
    elevator_bounds=elevator_bounds,
    cmy_tolerance=cmy_tolerance,
    max_trim_evaluations=max_trim_evaluations,
    trim_step_deg=trim_step_deg,
    analysis_method=analysis_method,
    ncpu=ncpu,
    wake_num_iter=wake_num_iter,
    fixed_wake_flag=fixed_wake_flag,
    verbose=verbose,
)

# 後段の抵抗補正が元DataFrameを書き換えないよう、未補正結果を保持する。
trimmed_polar_raw = trimmed_polar.copy()

print("rows:", len(trimmed_polar_raw))
print("columns:")
print(", ".join(trimmed_polar_raw.columns))
display(trimmed_polar_raw.head())

*

## 5. トリム収束の確認

空力係数そのものだけでなく、次を迎角ごとに確認する。

- `Elevator_deg`：受理されたエレベータ舵角
- `CMyResidual`：受理点の`CMytot`
- `CMyElevatorSlope_per_deg`：局所的なエレベータ有効度
- `TrimFunctionEvaluations`：その迎角でのトリム評価回数
- `SpeedConditionDuration_s`：速度条件を決める初回解析時間
- `TrimDuration_s`：エレベータトリムに使った解析時間の合計

In [ ]:
diagnostic_columns = [
    "InputAlpha_deg",
    "Elevator_deg",
    "CMyResidual",
    "CMyElevatorSlope_per_deg",
    "TrimFunctionEvaluations",
    "SpeedConditionDuration_s",
    "TrimDuration_s",
    "InputMach",
    "InputReCref",
]

display(trimmed_polar_raw[diagnostic_columns])

max_abs_cmy_residual = float(
    trimmed_polar_raw["CMyResidual"].abs().max()
)
min_elevator_deg = float(trimmed_polar_raw["Elevator_deg"].min())
max_elevator_deg = float(trimmed_polar_raw["Elevator_deg"].max())

trim_summary = pd.Series(
    {
        "max_abs_CMyResidual": max_abs_cmy_residual,
        "cmy_tolerance": cmy_tolerance,
        "max_TrimFunctionEvaluations": int(
            trimmed_polar_raw["TrimFunctionEvaluations"].max()
        ),
        "total_TrimFunctionEvaluations": int(
            trimmed_polar_raw["TrimFunctionEvaluations"].sum()
        ),
        "min_Elevator_deg": min_elevator_deg,
        "max_Elevator_deg": max_elevator_deg,
        "total_SpeedConditionDuration_s": float(
            trimmed_polar_raw["SpeedConditionDuration_s"].sum()
        ),
        "total_TrimDuration_s": float(
            trimmed_polar_raw["TrimDuration_s"].sum()
        ),
    },
    name="value",
)
display(trim_summary)

if max_abs_cmy_residual > cmy_tolerance:
    raise AssertionError(
        "An accepted polar row exceeds cmy_tolerance. "
        f"max_abs={max_abs_cmy_residual}, tolerance={cmy_tolerance}"
    )
if min_elevator_deg < elevator_bounds[0] or max_elevator_deg > elevator_bounds[1]:
    raise AssertionError(
        "An accepted elevator deflection lies outside elevator_bounds."
    )

## 6. 未補正polarの確認

`vsp_trimmed_sweep()`が返した未補正結果を確認する。

`Velocity`は`m/s`、`Vx`は`km/h`、`vz`は`m/s`である。`vz`は沈下速度の大きさとして正値で出力される。

In [ ]:
polar_columns = [
    "InputAlpha_deg",
    "Elevator_deg",
    "CL",
    "CDi",
    "CDtot",
    "L_D",
    "Velocity",
    "Vx",
    "vz",
]
available_polar_columns = [
    column
    for column in polar_columns
    if column in trimmed_polar_raw.columns
]

display(trimmed_polar_raw[available_polar_columns])

performance_summary = pd.Series(
    {
        "max_L_D": float(trimmed_polar_raw["L_D"].max()),
        "alpha_at_max_L_D_deg": float(
            trimmed_polar_raw.loc[
                trimmed_polar_raw["L_D"].idxmax(),
                "InputAlpha_deg",
            ]
        ),
        "min_sink_speed_mps": float(trimmed_polar_raw["vz"].min()),
        "Vx_at_min_sink_kmh": float(
            trimmed_polar_raw.loc[
                trimmed_polar_raw["vz"].idxmin(),
                "Vx",
            ]
        ),
    },
    name="value",
)
display(performance_summary)

## 7. トリム診断と滑空極のグラフ

図は一つずつ独立して作成し、Notebook表示と同時にPNGへ保存する。

In [ ]:
figure_paths = []

fig, ax = plt.subplots(figsize=(7.0, 4.5))
ax.plot(
    trimmed_polar_raw["InputAlpha_deg"],
    trimmed_polar_raw["Elevator_deg"],
    marker="o",
)
ax.set_xlabel("Angle of attack [deg]")
ax.set_ylabel("Trimmed elevator deflection [deg]")
ax.grid(True)
fig.tight_layout()
path = output_dir / "trimmed_elevator_vs_alpha.png"
fig.savefig(path, dpi=150)
figure_paths.append(path)
plt.show()

fig, ax = plt.subplots(figsize=(7.0, 4.5))
ax.semilogy(
    trimmed_polar_raw["InputAlpha_deg"],
    np.maximum(
        trimmed_polar_raw["CMyResidual"].abs(),
        np.finfo(float).tiny,
    ),
    marker="o",
)
ax.axhline(cmy_tolerance, linestyle="--")
ax.set_xlabel("Angle of attack [deg]")
ax.set_ylabel("Absolute CMytot residual")
ax.grid(True)
fig.tight_layout()
path = output_dir / "cmy_residual_vs_alpha.png"
fig.savefig(path, dpi=150)
figure_paths.append(path)
plt.show()

fig, ax = plt.subplots(figsize=(7.0, 4.5))
ax.plot(
    trimmed_polar_raw["InputAlpha_deg"],
    trimmed_polar_raw["TrimFunctionEvaluations"],
    marker="o",
)
ax.set_xlabel("Angle of attack [deg]")
ax.set_ylabel("Trim function evaluations")
ax.grid(True)
fig.tight_layout()
path = output_dir / "trim_evaluations_vs_alpha.png"
fig.savefig(path, dpi=150)
figure_paths.append(path)
plt.show()

fig, ax = plt.subplots(figsize=(7.0, 4.5))
ax.plot(
    trimmed_polar_raw["Vx"],
    trimmed_polar_raw["vz"],
    marker="o",
)
for row in trimmed_polar_raw.itertuples(index=False):
    ax.annotate(
        f"{row.InputAlpha_deg:g} deg",
        (row.Vx, row.vz),
        xytext=(5, 5),
        textcoords="offset points",
    )
ax.set_xlabel("Horizontal speed Vx [km/h]")
ax.set_ylabel("Sink speed vz [m/s]")
ax.invert_yaxis()
ax.grid(True)
fig.tight_layout()
path = output_dir / "uncorrected_glide_polar.png"
fig.savefig(path, dpi=150)
figure_paths.append(path)
plt.show()

for path in figure_paths:
    print(path)

## 8. 任意のCDo補正

`make_CDo_correction()`は、VSPAEROのpitch trimとは別の後処理である。`apply_cdo_correction=True`にした場合だけ、未補正DataFrameのコピーへ適用する。

補正値`xTr`、`CDpCL`、`thickness`、`interference_factor`は対象機に対する仮定であり、VSPAEROが自動的に決定する値ではない。

In [ ]:
corrected_polar = None

if apply_cdo_correction:
    corrected_polar = make_CDo_correction(
        vsp,
        trimmed_polar_raw.copy(),
        Weight=mass,
        altitude=altitude,
        dT=dT,
        **cdo_correction,
    )

    corrected_columns = [
        "InputAlpha_deg",
        "Elevator_deg",
        "CL",
        "CDi",
        "CDo_corr",
        "CDtot_corr",
        "L_D_corr",
        "Velocity",
        "Vx",
        "vz",
    ]
    display(corrected_polar[corrected_columns])

    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    ax.plot(
        trimmed_polar_raw["Vx"],
        trimmed_polar_raw["vz"],
        marker="o",
        label="VSPAERO uncorrected",
    )
    ax.plot(
        corrected_polar["Vx"],
        corrected_polar["vz"],
        marker="o",
        label="CDo corrected",
    )
    ax.set_xlabel("Horizontal speed Vx [km/h]")
    ax.set_ylabel("Sink speed vz [m/s]")
    ax.invert_yaxis()
    ax.grid(True)
    ax.legend()
    fig.tight_layout()
    corrected_plot_path = output_dir / "corrected_glide_polar.png"
    fig.savefig(corrected_plot_path, dpi=150)
    plt.show()
    print(corrected_plot_path)
else:
    print(
        "CDo correction was not applied. "
        "Set apply_cdo_correction=True and rerun from section 2 "
        "when the correction assumptions are intended."
    )

## 9. CSV出力

未補正polarは必ず保存する。CDo補正を実行した場合だけ、補正後polarを別ファイルへ保存する。

In [ ]:
raw_csv_path = output_dir / "G103A_trimmed_polar_raw.csv"
trimmed_polar_raw.to_csv(raw_csv_path, index=False)
print("raw_csv_path:", raw_csv_path)

corrected_csv_path = None
if corrected_polar is not None:
    corrected_csv_path = output_dir / "G103A_trimmed_polar_corrected.csv"
    corrected_polar.to_csv(corrected_csv_path, index=False)
    print("corrected_csv_path:", corrected_csv_path)

## 10. 確認事項

- OpenVSP Python環境とVSPAERO実行環境が利用可能であること。
- 読み込んだモデルに`WingGeom`がちょうど1個存在すること。
- `ELEVATOR_GROUP`がちょうど1個存在し、有効なControl Surfaceを持つこと。
- `Control Group Angles`で`ELEVATOR_GROUP`がONになっていること。
- `VSPAEROComputeGeometry`のThick/Thin geometry setが対象モデルの意図どおりであること。
- 全caseで`abs(CMyResidual) <= cmy_tolerance`となっていること。
- `Elevator_deg`が`elevator_bounds`の端へ張り付いていないこと。
- `CMyElevatorSlope_per_deg`の符号と大きさが迎角間で不自然に急変していないこと。
- 後続caseで`TrimFunctionEvaluations`が減るか、少なくとも過度に増加していないこと。
- `CL > 0`であり、`Velocity`、`L_D`、`Vx`、`vz`が有限値であること。
- 未補正結果と経験的なCDo補正後の結果を混同しないこと。
- 迎角列を増やす前に、少数点で計算時間とwake収束設定を確認すること。
- `vsp_trimmed_sweep()`終了後に、モデルの元のエレベータ舵角へ復元されていること。